In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


#  DL & GenAI — MILESTONE 1

In [2]:
# import
import string
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Load
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print("Columns:", train.columns.tolist())
print("Shape  :", train.shape)

Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Shape  : (2000, 8)


In [4]:
#  Column config
ID, PROMPT, ANSWER = "id", "prompt", "answer"
OPTIONS = ["A", "B", "C", "D", "E"]

In [5]:
# Helpers 
def clean_text(t):
    """lowercase + remove string.punctuation (chars are deleted, not spaced)."""
    t = str(t).lower()
    return t.translate(str.maketrans("", "", string.punctuation))

def map3(truth, preds):
    """MAP@3 for a single question (one correct label)."""
    preds = list(preds)[:3]
    for i, p in enumerate(preds):
        if p == truth:
            return 1.0 / (i + 1)       
    return 0.0

**Q1: Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?**

In [6]:
vc = train[ANSWER].value_counts()
print("\nDistribution:\n", vc)
Q1 = int(vc.max() + vc.min())
print("Q1:",Q1)


Distribution:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Q1: 814


**Q2: After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?**

In [7]:
vocab = set()
for p in train[PROMPT]:
    vocab.update(clean_text(p).split())
Q2 = len(vocab)
print("Q2:",Q2)

Q2: 859


**Q3: Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?**

In [8]:
row1 = train[train[ID] == 1].iloc[0] if (train[ID] == 1).any() else train.iloc[1]
toks_r1 = clean_text(row1[PROMPT]).split()
Q3 = len([w for w in toks_r1 if w not in ENGLISH_STOP_WORDS])
print("Q3:",Q3)

Q3: 13


**Q4: Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?**

In [9]:
corpus = train[PROMPT].astype(str).tolist()
for c in OPTIONS:
    corpus += train[c].astype(str).tolist()
tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(corpus)
Q4 = len(tfidf.get_feature_names_out())
print("Q4:",Q4)

Q4: 2762


**Q5: Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).**

In [10]:
pv = tfidf.transform([str(row1[PROMPT])])
av = tfidf.transform([str(row1["A"])])
Q5 = round(float(cosine_similarity(pv, av)[0, 0]), 4)
print("Q5:",Q5)

Q5: 0.2328


**Q6: Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.**

In [11]:
correct = 0
for _, r in train.iterrows():
    pv = tfidf.transform([str(r[PROMPT])])
    ov = tfidf.transform([str(r[c]) for c in OPTIONS])
    sims = cosine_similarity(pv, ov).ravel()
    if OPTIONS[int(np.argmax(sims))] == r[ANSWER]:
        correct += 1
Q6 = round(100 * correct / len(train), 2)
print("Q6:",Q6)

Q6: 13.7


**Q7: If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?**

**Q8: If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?**

In [12]:
Q7 = map3("C", ["C", "A", "B"])         
Q8 = map3("B", ["D", "B", "E"])
print("Q7:",Q7)
print("Q8:",Q8)

Q7: 1.0
Q8: 0.5


**Q9: The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?**

In [13]:
order = train[ANSWER].value_counts().index.tolist()
top3_static = order[:3]
Q9 = round(float(np.mean([map3(t, top3_static) for t in train[ANSWER]])), 4)
print("Q9:",Q9)

Q9: 0.4213


**Q10: The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?**

In [14]:
maps = []
for _, r in train.iterrows():
    pv = tfidf.transform([str(r[PROMPT])])
    ov = tfidf.transform([str(r[c]) for c in OPTIONS])
    sims = cosine_similarity(pv, ov).ravel()
    ranked = [OPTIONS[i] for i in np.argsort(sims)[::-1]]   # high -> low
    maps.append(map3(r[ANSWER], ranked))
Q10 = round(float(np.mean(maps)), 4)
print("Q10:",Q10)

Q10: 0.2762


In [15]:
print("MILESTONE 1 - FORM ANSWERS")
for k, v in dict(Q1=Q1, Q2=Q2, Q3=Q3, Q4=Q4, Q5=Q5,
                 Q6=Q6, Q7=Q7, Q8=Q8, Q9=Q9, Q10=Q10).items():
    print(f"{k}: {v}")

MILESTONE 1 - FORM ANSWERS
Q1: 814
Q2: 859
Q3: 13
Q4: 2762
Q5: 0.2328
Q6: 13.7
Q7: 1.0
Q8: 0.5
Q9: 0.4213
Q10: 0.2762


In [16]:
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

PROJECT = "23f1002033-t22026"

# Run 1: Majority-class baseline
run = wandb.init(entity="ishankgpt02-na", project="23f1002033-t22026",
                 name="m1-majority-baseline",
                 config={"milestone": 1, "approach": "majority_class", "metric": "map@3"})
wandb.log({"map@3": Q9})
run.finish()

# Run 2: TF-IDF cosine baseline
run = wandb.init(entity="ishankgpt02-na", project="23f1002033-t22026",
                 name="m1-tfidf-baseline",
                 config={"milestone": 1, "approach": "tfidf_cosine", "metric": "map@3"})
wandb.log({"map@3": Q10, "top1_match_pct": Q6})
run.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ishankgpt02 (ishankgpt02-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260624_110445-drdploho
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run m1-majority-baseline
wandb: ⭐️ View project at https://wandb.ai/ishankgpt02-na/23f1002033-t22026
wandb: 🚀 View run at https://wandb.ai/ishankgpt02-na/23f1002033-t22026/runs/drdploho
wandb: updating run metadata; uploading summary
wandb: uploading history